# A tiny neuroscience lab in your browser

**NumPy · xarray · h5netcdf · pandas · h5py · Neo · Elephant · seaborn · Pingouin**

Simulate paired neural recordings, analyze spike trains, save real HDF5 files, and compare two conditions. Everything runs locally in your browser with JupyterLite / Xeus-Python (xpython)—no Python server. The same notebook also runs in a regular Python Jupyter kernel.

Choose **Run → Run All Cells**. The first launch downloads the prebuilt Python environment; packages are already installed, with no browser-side solve or installation. An internet connection is needed for initial loading. Each cell can also be run with **Shift+Enter**.

This is synthetic teaching data, not evidence for a biological effect. Files live in a virtual filesystem; use the download links at the end to keep them.

## 1a. Environment setup (separate from imports)

The deployment installs the WebAssembly packages **once at build time** using micromamba and `environment.yml` (emscripten-forge + conda-forge), then bundles the environment with Xeus-Python. This cell intentionally performs no installation. Native Python users should run `pixi install --locked` before opening the notebook.

Elephant **1.2.0** is the one pip exception: it has an official pure-Python wheel, but no Wasm conda build. Its dependencies come from mamba. Its real analysis functions are used; optional native/GPU accelerators are outside this demo.

In [ ]:
# Installation is defined in environment.yml and performed during site build.
# No pip/mamba command is needed when opening or rerunning this notebook.
print('Setup: using the preinstalled scientific environment.')

## 1b. Import the scientific stack

Imports and package versions are separate from environment setup, so you can rerun this cell without reinstalling anything.

In [ ]:
%matplotlib inline

import sys
from importlib.metadata import version
from pathlib import Path
import base64

import numpy as np
import xarray as xr
import h5netcdf
import pandas as pd
import h5py
import neo
import elephant
import quantities as pq
import seaborn as sns
import pingouin as pg
import matplotlib.pyplot as plt
from elephant.statistics import mean_firing_rate, isi, cv
from elephant.conversion import BinnedSpikeTrain
from IPython.display import HTML, display

packages = ['numpy', 'xarray', 'h5netcdf', 'pandas', 'h5py',
            'neo', 'elephant', 'seaborn', 'pingouin']
display(pd.DataFrame({'package': packages, 'version': [version(p) for p in packages]}))
print('Runtime:', 'Xeus-Python / WebAssembly' if sys.platform == 'emscripten' else 'native Python')

## 2. Simulate paired recordings — change these parameters!

Each neuron is recorded in **baseline** and **stimulus** conditions over repeated trials. Counts are Poisson-distributed; conditional on the count, spike times are uniform in the recording interval (a homogeneous Poisson process). NumPy generates the data and Neo's `SpikeTrain` attaches time units and recording bounds.

Try setting `stimulus_gain_hz = 0.0`, changing `seed`, or increasing `n_trials`. Rerun this cell and everything below it. Keep the sizes modest for a browser.

In [ ]:
seed = 42
n_neurons = 12
n_trials = 8
duration_s = 2.0
baseline_rate_hz = 15.0
stimulus_gain_hz = 7.0
bin_size_ms = 100.0

assert n_neurons >= 3 and n_trials >= 1
assert duration_s > 0 and baseline_rate_hz > 0 and bin_size_ms > 0
assert np.isclose(duration_s * 1000 / bin_size_ms, round(duration_s * 1000 / bin_size_ms)), 'Choose bins that divide the duration'
rng = np.random.default_rng(seed)
conditions = ['baseline', 'stimulus']
neuron_baselines = np.clip(rng.normal(baseline_rate_hz, 2.0, n_neurons), 0.5, None)
spike_trains = {}
records = []
rates = np.empty((n_neurons, len(conditions), n_trials))

for neuron_id in range(n_neurons):
    for condition_id, condition in enumerate(conditions):
        target_hz = max(0.5, neuron_baselines[neuron_id] + condition_id * stimulus_gain_hz)
        for trial in range(n_trials):
            count = rng.poisson(target_hz * duration_s)
            times = np.sort(rng.uniform(0, duration_s, count))
            train = neo.SpikeTrain(times * pq.s, t_start=0 * pq.s, t_stop=duration_s * pq.s)
            spike_trains[neuron_id, condition, trial] = train
            rate_hz = float(mean_firing_rate(train).rescale(pq.Hz).magnitude)
            intervals = isi(train)
            isi_cv = float(cv(intervals)) if len(intervals) >= 2 else np.nan
            rates[neuron_id, condition_id, trial] = rate_hz
            records.append({'neuron': neuron_id, 'condition': condition, 'trial': trial,
                            'rate_hz': rate_hz, 'isi_cv': isi_cv, 'spike_count': count})

trials = pd.DataFrame.from_records(records)
assert len(trials) == n_neurons * 2 * n_trials
np.testing.assert_allclose(trials['rate_hz'], trials['spike_count'] / duration_s)
display(trials.head())
print(f'{len(trials)} Neo spike trains; {trials.spike_count.sum()} total spikes')

## 3. Label the data with xarray; summarize with pandas

The rate tensor has named `neuron`, `condition`, and `trial` dimensions. Trial-averaged rates become one paired observation per neuron. This avoids treating repeated trials as independent neurons.

In [ ]:
dataset = xr.Dataset(
    {'firing_rate': (('neuron', 'condition', 'trial'), rates)},
    coords={'neuron': np.arange(n_neurons), 'condition': conditions, 'trial': np.arange(n_trials)},
    attrs={'description': 'Synthetic paired spike-train recordings', 'seed': seed, 'duration_s': duration_s},
)
dataset['firing_rate'].attrs['units'] = 'Hz'
mean_rates = dataset.firing_rate.mean('trial')
paired = mean_rates.to_pandas()
paired_long = paired.reset_index().melt(id_vars='neuron', var_name='condition', value_name='rate_hz')
summary = trials.groupby('condition', sort=False).agg(
    mean_rate_hz=('rate_hz', 'mean'), mean_isi_cv=('isi_cv', 'mean'), total_spikes=('spike_count', 'sum')
)
display(dataset)
display(summary)

## 4. Real HDF5 round-trips: xarray + h5netcdf + h5py

Write the labeled dataset as NetCDF4/HDF5 using the **h5netcdf** engine. Read it back with xarray and inspect the dimensions directly with h5netcdf. Separately, use **h5py** to store the variable-length spike trains as individual HDF5 datasets, including unit and recording-duration metadata. These operations also run in the browser.

In [ ]:
netcdf_path = Path('demo_recordings.nc')
spikes_path = Path('demo_spikes.h5')
dataset.to_netcdf(netcdf_path, engine='h5netcdf', mode='w')
with xr.open_dataset(netcdf_path, engine='h5netcdf') as opened:
    restored = opened.load()
xr.testing.assert_identical(dataset, restored)

with h5netcdf.File(netcdf_path, 'r') as file:
    assert len(file.dimensions['neuron']) == n_neurons
    assert file.variables['firing_rate'].attrs['units'] == 'Hz'
    np.testing.assert_allclose(file.variables['firing_rate'][:], rates)
    print('h5netcdf dimensions:', {name: len(dim) for name, dim in file.dimensions.items()})

with h5py.File(spikes_path, 'w') as file:
    file.attrs['description'] = 'Synthetic spike times; one dataset per trial'
    file.attrs['duration_s'] = duration_s
    for (neuron_id, condition, trial), train in spike_trains.items():
        group = file.require_group(f'neuron_{neuron_id}/{condition}')
        stored = group.create_dataset(f'trial_{trial}', data=train.rescale(pq.s).magnitude)
        stored.attrs['units'] = 's'

with h5py.File(spikes_path, 'r') as file:
    first = file['neuron_0/baseline/trial_0']
    assert first.attrs['units'] == 's'
    recovered_train = neo.SpikeTrain(first[:] * pq.s, t_stop=file.attrs['duration_s'] * pq.s)
    np.testing.assert_array_equal(recovered_train.magnitude, spike_trains[0, 'baseline', 0].magnitude)

assert h5py.is_hdf5(netcdf_path) and h5py.is_hdf5(spikes_path)
print('Both HDF5 round-trips passed:', netcdf_path.name, 'and', spikes_path.name)

## 5. Paired inference with Pingouin

Compare stimulus and baseline **within neurons** using a two-sided paired t-test. The test assumes independent neurons and approximately normal within-neuron differences. Inspect the difference distribution for real analyses, and account for multiple comparisons when testing many hypotheses. Here the mean difference and its t-based 95% confidence interval are in Hz. Pingouin also reports an effect size and other diagnostics.

In [ ]:
comparison = pg.ttest(paired['stimulus'], paired['baseline'], paired=True, alternative='two-sided')
difference = paired['stimulus'] - paired['baseline']
assert int(comparison['dof'].iloc[0]) == n_neurons - 1
assert np.isfinite(comparison['T'].iloc[0])
display(comparison)
print(f'Mean within-neuron difference: {difference.mean():.2f} Hz (n={n_neurons} paired neurons)')

## 6. Visualize with seaborn and Elephant binning

A raster shows one neuron's trials. Each line in the middle joins one neuron's trial-averaged rates. On the right, Elephant's `BinnedSpikeTrain` turns the baseline trials into a count matrix displayed with seaborn.

In [ ]:
sns.set_theme(style='whitegrid', context='notebook')
baseline_trains = [spike_trains[0, 'baseline', trial] for trial in range(n_trials)]
binned = BinnedSpikeTrain(baseline_trains, bin_size=bin_size_ms * pq.ms)
counts = binned.to_array()
assert counts.shape == (n_trials, round(duration_s * 1000 / bin_size_ms))
np.testing.assert_array_equal(counts.sum(axis=1), [len(train) for train in baseline_trains])

fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
palette = sns.color_palette('colorblind', 2)
for condition_id, condition in enumerate(conditions):
    for trial in range(n_trials):
        times = spike_trains[0, condition, trial].rescale(pq.s).magnitude
        axes[0].scatter(times, np.full(len(times), trial + condition_id * n_trials),
                        marker='|', s=28, color=palette[condition_id],
                        label=condition if trial == 0 else None)
axes[0].set(xlabel='Time (s)', ylabel='Trial (baseline, then stimulus)', title='Neuron 0: spike raster', xlim=(0, duration_s))
axes[0].legend(fontsize='small')
sns.lineplot(data=paired_long, x='condition', y='rate_hz', units='neuron', estimator=None,
             marker='o', alpha=0.65, color=palette[0], ax=axes[1])
axes[1].set(xlabel='', ylabel='Trial-averaged rate (Hz)', title='Paired neurons')
sns.heatmap(counts, cmap='mako', cbar_kws={'label': 'Spike count'}, ax=axes[2])
axes[2].set(xlabel=f'Time bin ({bin_size_ms:g} ms)', ylabel='Baseline trial', title='Elephant binned spikes')
plt.show()

## 7. Download your results

These links embed the generated bytes, so they work without a Python server or a shared host filesystem. The NetCDF file can be reopened with `xr.open_dataset(path, engine='h5netcdf')`. Download your edited notebook using **File → Save and Export Notebook As** as well.

In [ ]:
for path in [netcdf_path, spikes_path]:
    encoded = base64.b64encode(path.read_bytes()).decode('ascii')
    display(HTML(f'<a download="{path.name}" href="data:application/octet-stream;base64,{encoded}">Download {path.name}</a>'))

assert elephant.__version__ == '1.2.0'
assert restored.firing_rate.shape == rates.shape
assert len(paired) == n_neurons and comparison.shape[0] == 1
print('DEMO_COMPLETE: all nine requested libraries exercised.')